In [1]:
# Parameters
input_file = "/home/shiva/WMU/PhD/Scripts/phd_project/Python/NMR/CASTEP/L-alanine_295K_278464_17O_opt_rPBE_magres.magres"


This code uses papermill on terminal to change input_file. The input_file is the file path to the magres file

*Make sure you change the nucleus information like Q value and nucleus (I am not sure how to do this in an efficient way)*

Shiva Agarwal

*Apr 23 2025*

In [2]:
import os
import numpy as np
from tabulate import tabulate
from magres.atoms import MagresAtoms

atoms = MagresAtoms.load_magres(input_file)


In [3]:
def Rabc(alfa1, beta1, gama1): #Euler Rotation Matrix
    U = np.zeros((3, 3))
    #Changing input angles from degrees to radians
    alfa1 = np.radians(alfa1) 
    beta1 = np.radians(beta1)
    gama1 = np.radians(gama1)
    
    U[0, 0] = np.cos(alfa1)*np.cos(beta1)*np.cos(gama1) - np.sin(alfa1)*np.sin(gama1)
    U[0, 1] = np.sin(alfa1)*np.cos(beta1)*np.cos(gama1) + np.cos(alfa1)*np.sin(gama1)
    U[0, 2] = -np.sin(beta1)*np.cos(gama1)
    U[1, 0] = -np.cos(alfa1)*np.cos(beta1)*np.sin(gama1) - np.sin(alfa1)*np.cos(gama1)
    U[1, 1] = -np.sin(alfa1)*np.cos(beta1)*np.sin(gama1) + np.cos(alfa1)*np.cos(gama1)
    U[1, 2] = np.sin(beta1)*np.sin(gama1)
    U[2, 0] = np.cos(alfa1)*np.sin(beta1)
    U[2, 1] = np.sin(alfa1)*np.sin(beta1)
    U[2, 2] = np.cos(beta1)
    return U

In [4]:
# Sort eigenvalues of tensors as per the convention defined in article
def sort_eigenvalues(Tensor):
    #Calculate Quadrupolar Tensor in PAS
    eigenvalues, eigenvectors = np.linalg.eig(Tensor)
    print(' Unsorted Eigenvalues:\n', eigenvalues, '\n')
    print(' Unsorted Eigenvectors:\n', eigenvectors, '\n')

    avg_tensor = np.mean(eigenvalues) # Tr(A)Quad/3
    eigenvalue_diff = eigenvalues - avg_tensor

    # Get the indices of the sorted eigenvalues based on the absolute values
    sorted_indices = np.argsort(np.abs(eigenvalue_diff))

    # Sort both eigenvalues and eigenvectors using the sorted indices
    sorted_eigenvalues = eigenvalues[sorted_indices]
    sorted_eigenvectors = eigenvectors[:, sorted_indices] # The normalized (unit “length”) eigenvectors, 
    #                                                       such that the column eigenvectors[:,i] is the eigenvector corresponding to the eigenvalue eigenvalues[i]
    # eigenvectors need to be arranged so that first column for direction cosine matrix is eigenvector for x, second column is for y and third column is for z
    y_dc = sorted_eigenvectors[:,0]
    x_dc = sorted_eigenvectors[:,1]
    z_dc = sorted_eigenvectors[:,2]

    dc = np.stack((x_dc, y_dc, z_dc), axis = 1)


    print('Sorted Eigenvalues: \n', sorted_eigenvalues, '\n')
    print('Sorted Eigenvectors: \n', sorted_eigenvectors, '\n')
    return sorted_eigenvalues, dc,  avg_tensor, eigenvalues, eigenvectors

In [5]:
def get_euler_angles(eigenvectors):
    b = np.degrees(np.arccos(eigenvectors[2,2]))
    a = np.degrees(np.arctan(eigenvectors[2,1]/eigenvectors[2,0]))
    g = np.degrees(np.arctan(-eigenvectors[1,2]/eigenvectors[0,2]))

    return a, b, g
    

In [6]:
nucleus = 'O'      # nucleus for which parameters are wanted
atom_label = 0      # site for which parameters wanted
Q = -0.0256         #electric quadrupole moment for nucleus in barn

In [7]:
for atom in atoms.species(nucleus):
    print (atom, "sigma:\n",atom.ms.sigma)
    print()

17O1 sigma:
 [[-47.54524348 167.97681205 -81.64300162]
 [175.04236919  57.68488635  30.2326934 ]
 [  8.13413673 -24.24537172 -46.58115879]]

17O2 sigma:
 [[ -47.54524348 -167.97681205   81.64300162]
 [-175.04236919   57.68488635   30.2326934 ]
 [  -8.13413673  -24.24537172  -46.58115879]]

17O3 sigma:
 [[-47.54524348 167.97681205  81.64300162]
 [175.04236919  57.68488635 -30.2326934 ]
 [ -8.13413673  24.24537172 -46.58115879]]

17O4 sigma:
 [[ -47.54524348 -167.97681205  -81.64300162]
 [-175.04236919   57.68488635  -30.2326934 ]
 [   8.13413673   24.24537172  -46.58115879]]

17O5 sigma:
 [[ -36.79380755  224.74015721   54.03920372]
 [ 195.07990097  107.5701716   -87.10338316]
 [  13.31718655  -47.55113453 -173.23524624]]

17O6 sigma:
 [[ -36.79380755 -224.74015721  -54.03920372]
 [-195.07990097  107.5701716   -87.10338316]
 [ -13.31718655  -47.55113453 -173.23524624]]

17O7 sigma:
 [[ -36.79380755  224.74015721  -54.03920372]
 [ 195.07990097  107.5701716    87.10338316]
 [ -13.31718655

In [8]:
for atom in atoms.species('O'):
    print (atom, "sigma:\n",atom.efg.Cq)
    print()

17O1 sigma:
 6.532497666502127

17O2 sigma:
 6.532497666502117

17O3 sigma:
 6.532497666502155

17O4 sigma:
 6.53249766650215

17O5 sigma:
 8.59292669772807

17O6 sigma:
 8.592926697728068

17O7 sigma:
 8.592926697728082

17O8 sigma:
 8.592926697728082



In [9]:
# using values from latest magres file
Cs = np.zeros((3, 3))                                # CS symmetric (l = 0 + 2) Tensor from updated_magres
CS_anti = np.zeros((3,3))                           # CS antisymmetric ( l = 1)
CS_iso = np.zeros((3,3))                            # CS isotropic  (l = 0)
CS_total = np.zeros((3,3))                            # CS total shielding tensor ( l = 0 + 1 + 2) Tensor from magres

                                         
CS_total[:,:] = atoms.species(nucleus).ms.sigma[atom_label]

iso = np.mean([CS_total[0,0], CS_total[1,1], CS_total[2,2]]) # isotropic chemical shielding (l = 0)

CS_iso[0,0] = CS_iso[1,1] = CS_iso[2,2] = iso

Cs[0,0] = CS_total[0,0]; Cs[0,1] = (CS_total[0,1] + CS_total[1,0] )/2; Cs[0,2] = (CS_total[0,2] + CS_total[2,0])/2;
Cs[1,0] = Cs[0,1];      Cs[1,1] = CS_total[1,1];                      Cs[1,2] = (CS_total[1,2] + CS_total[2,1])/2;
Cs[2,0] = Cs[0,2];      Cs[2,1] = Cs[1,2];                           Cs[2,2] = CS_total[2,2];

CS_anti[0,1] = (CS_total[0,1] - CS_total[1,0])/2; CS_anti[0,2] = (CS_total[0,2] - CS_total[2,0])/2; 
CS_anti[1,0] = -CS_anti[0,1]; CS_anti[1,2] = (CS_total[1,2] - CS_total[2,1])/2;
CS_anti[2,0] = -CS_anti[0,2];      CS_anti[2,1] = -CS_anti[1,2]; 


efg = np.zeros((3, 3 ))                             # EFG Tensor from magres (in a.u.)

efg[:,:] = atoms.species(nucleus)[atom_label].efg.V


# Convert a.u. units to MHz
# Q tensor elements (MHz) = efg tensor (a.u.)* Q (barn) * 234.9647 
# Q = 0.04059 barn https://www-nds.iaea.org/publications/indc/indc-nds-0650.pdf

V = efg*Q*234.9647

print('\nQ tensor:\n', np.round(V,3))
print('\nCS Tensor:\n',np.round(CS_total, 3))
print('\nCS isotropic Tensor:\n',np.round(CS_iso, 3))
print('\nCS symmetric Tensor:\n',np.round(Cs,3))
print('\nCS antisymmetric Tensor:\n',np.round(CS_anti,3))



Q tensor:
 [[ 0.941 -0.798  4.685]
 [-0.798 -0.399 -3.974]
 [ 4.685 -3.974 -0.543]]

CS Tensor:
 [[-47.545 167.977 -81.643]
 [175.042  57.685  30.233]
 [  8.134 -24.245 -46.581]]

CS isotropic Tensor:
 [[-12.147   0.      0.   ]
 [  0.    -12.147   0.   ]
 [  0.      0.    -12.147]]

CS symmetric Tensor:
 [[-47.545 171.51  -36.754]
 [171.51   57.685   2.994]
 [-36.754   2.994 -46.581]]

CS antisymmetric Tensor:
 [[  0.     -3.533 -44.889]
 [  3.533   0.     27.239]
 [ 44.889 -27.239   0.   ]]


In [10]:
print("For EFG tensor")
sorted_eigenvalues_efg, dc_efg, quad_avg, eigenvalues_efg, eigenvectors_efg = sort_eigenvalues(V)
print('==================================\n')
print("For CS tensor")
sorted_eigenvalues_cs, dc_cs, cs_avg, eigenvalues_cs, eigenvectors_cs = sort_eigenvalues(Cs)

For EFG tensor
 Unsorted Eigenvalues:
 [ 6.53760293 -0.62428618 -5.91331675] 

 Unsorted Eigenvectors:
 [[-0.61122568  0.64539297 -0.45811689]
 [ 0.44524694  0.7589436   0.47514184]
 [-0.65433808 -0.08644375  0.75124507]] 

Sorted Eigenvalues: 
 [-0.62428618 -5.91331675  6.53760293] 

Sorted Eigenvectors: 
 [[ 0.64539297 -0.45811689 -0.61122568]
 [ 0.7589436   0.47514184  0.44524694]
 [-0.08644375  0.75124507 -0.65433808]] 


For CS tensor
 Unsorted Eigenvalues:
 [ 186.11197749 -181.65424216  -40.89925125] 

 Unsorted Eigenvectors:
 [[ 0.5981986   0.79015671 -0.13345712]
 [ 0.7969084  -0.56907065  0.20272051]
 [-0.08423444  0.22762023  0.97009978]] 

Sorted Eigenvalues: 
 [ -40.89925125 -181.65424216  186.11197749] 

Sorted Eigenvectors: 
 [[-0.13345712  0.79015671  0.5981986 ]
 [ 0.20272051 -0.56907065  0.7969084 ]
 [ 0.97009978  0.22762023 -0.08423444]] 



In [11]:

#Calculate Quadrupolar Tensor in PAS

Vyy = sorted_eigenvalues_efg[0]
Vxx = sorted_eigenvalues_efg[1]
Vzz = sorted_eigenvalues_efg[2]

print('Quadupolar Tensor Components Vyy, Vxx, Vzz: \n', Vyy, Vxx, Vzz)

print('================================================================================================')

#Calculate CSA Tensor in PAS

Csyy = sorted_eigenvalues_cs[0] 
Csxx = sorted_eigenvalues_cs[1]  
Cszz = sorted_eigenvalues_cs[2]



print('CSA Tensor Components δyy, δxx, δzz: \n', Csyy, Csxx, Cszz)

Quadupolar Tensor Components Vyy, Vxx, Vzz: 
 -0.6242861794624951 -5.913316751187908 6.537602930650383
CSA Tensor Components δyy, δxx, δzz: 
 -40.899251249219546 -181.65424216070858 186.11197749344117


In [12]:
iso_cs = (Csxx + Csyy + Cszz)/3

csa = Cszz - iso_cs
etas = (Csyy - Csxx)/csa

#for Quadrupolar
CQ_fit = Vzz

etaq = (Vyy - Vxx)/Vzz

table = [['CQ (MHz)', CQ_fit], ['etaq', etaq ], ['iso_cs (ppm)',iso_cs ],['csa (ppm)', csa],  ['etas', etas]  ]

table_string = tabulate(table, headers=['Quantity', 'Value'], tablefmt='grid')
print(f'Parameters for {atoms.species(nucleus)[atom_label]}: \n', table_string)

Parameters for 17O1: 
 +--------------+------------+
| Quantity     |      Value |
+==============+============+
| CQ (MHz)     |   6.5376   |
+--------------+------------+
| etaq         |   0.809017 |
+--------------+------------+
| iso_cs (ppm) | -12.1472   |
+--------------+------------+
| csa (ppm)    | 198.259    |
+--------------+------------+
| etas         |   0.709955 |
+--------------+------------+


In [13]:

# Derive output name from input file
base_name = os.path.splitext(os.path.basename(input_file))[0]
output_txt = f"/home/shiva/WMU/PhD/Scripts/phd_project/Python/NMR/NMR_analysis/output_txt/{nucleus}_all_results.txt"

# Save to .txt file
with open(output_txt, 'a') as f:
    f.write(f"\n\n===== Results for: {base_name} =====\n\n")
    f.write(table_string)
    f.write("\n")

print(f"Saved table to {output_txt}")

Saved table to /home/shiva/WMU/PhD/Scripts/phd_project/Python/NMR/NMR_analysis/output_txt/O_all_results.txt


In [14]:
# Calculation for efg tensor
print('Direction cosine efg:\n')
print(dc_efg, '\n')
a_efg, b_efg, g_efg = get_euler_angles(dc_efg)

print("Calculated Euler angles (degrees) Quadrupolar PAS --> Crystal:")
print(a_efg, b_efg, g_efg, '\n')

print('=========================')
print('Direction cosine csa: \n')
print(dc_cs, '\n')
a_cs, b_cs, g_cs = get_euler_angles(dc_cs)

print("Calculated Euler angles (degrees) CSA PAS --> Crystal:")
print(a_cs, b_cs, g_cs, '\n')

Direction cosine efg:

[[-0.45811689  0.64539297 -0.61122568]
 [ 0.47514184  0.7589436   0.44524694]
 [ 0.75124507 -0.08644375 -0.65433808]] 

Calculated Euler angles (degrees) Quadrupolar PAS --> Crystal:
-6.56400292733379 130.86947897716914 36.07144165699319 

Direction cosine csa: 

[[ 0.79015671 -0.13345712  0.5981986 ]
 [-0.56907065  0.20272051  0.7969084 ]
 [ 0.22762023  0.97009978 -0.08423444]] 

Calculated Euler angles (degrees) CSA PAS --> Crystal:
76.79522045196899 94.83200375594656 -53.10630647727991 



In [15]:
# Euler Matrix to relate Quadrupolar and CSA tensor

CSA_Q = np.matmul(np.linalg.inv(dc_efg), (dc_cs))

psi, chi, xi = get_euler_angles(CSA_Q)

print("Calculated Euler angles (degrees) PAS CSA --> Quadrupole:")
print('psi:', psi, 'chi:', chi, 'xi:', xi, '\n')

Calculated Euler angles (degrees) PAS CSA --> Quadrupole:
psi: 27.606407882590187 chi: 87.46070879162968 xi: -87.62959632867236 

